**REGRESION LINEAL**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("segmentos_con_alcaldia_y_promedio.csv")
df["hora_exacta"] = pd.to_datetime(df["hora_exacta"])
df["hora_decimal"] = (df["hora_exacta"].dt.hour + df["hora_exacta"].dt.minute / 60)

df["sentido"] = df["sentido"].map({"Ida": 0, "Vuelta": 1})
df = pd.get_dummies(df, columns=["segmento_id", "dia_semana"], drop_first=True)


cols_to_drop = [
    "hora_exacta",
    "tiempo_segmento",
    "tiempo_acumulado_viaje",
    "vel_promedio_movimiento",
    "Hora"]
X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
y = df["tiempo_segmento"]

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

cv_results = cross_validate(
    pipeline, X, y,
    cv=kf,
    scoring=('neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2'),
    return_estimator=True)

print(" RESULTADOS K-FOLD (10 PLIEGUES) ")

print(f"MAE  : {-cv_results['test_neg_mean_absolute_error'].mean():.2f} segundos")
print(f"RMSE : {-cv_results['test_neg_root_mean_squared_error'].mean():.2f} segundos")
print(f"R²   : {cv_results['test_r2'].mean():.3f}")

coefs = np.array([est.named_steps['regressor'].coef_ for est in cv_results['estimator']])
mean_coefs = coefs.mean(axis=0)

coeficientes_df = pd.DataFrame({"Variable": X.columns, "Coeficiente": mean_coefs})
coeficientes_df = coeficientes_df.sort_values(by="Coeficiente", key=abs, ascending=False)
print(" IMPORTANCIA DE VARIABLES (COEFICIENTES)")
print(coeficientes_df.head(10))

### Matriz de Correlación de las Variables Más Importantes

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

top_variables = coeficientes_df['Variable'].head(10).tolist()

if 'tiempo_segmento' not in top_variables:
    top_variables.append('tiempo_segmento')

filtrado = df[top_variables].copy()
correlacion_importantes = filtrado.corr(numeric_only=True)

plt.figure(figsize=(12, 10))
mask = np.triu(correlacion_importantes)
sns.heatmap(correlacion_importantes, annot=True, cmap="coolwarm", fmt=".2f",
            mask=mask, center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})

plt.title("Matriz de Correlación de las Variables Más Importantes")
plt.tight_layout()
plt.show()

**Gráfica del Regresión Lineal Multiple**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pipeline.fit(X, y)
y_pred_total = pipeline.predict(X)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y, y=y_pred_total, alpha=0.5, color='royalblue')
max_val = max(y.max(), y_pred_total.max())
min_val = min(y.min(), y_pred_total.min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', lw=2, label='Predicción Perfecta')

plt.title('Evaluación del Modelo: Tiempo Real vs. Predicho (Regresión Lineal)')
plt.xlabel('Tiempo Real (segundos)')
plt.ylabel('Tiempo Predicho (segundos)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

**RED NEURONAL **(atendiendo sus indicaciones Dra)

**Red Neuronal Perceptrón Multicapa - KFold Validation 10**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

df = pd.read_csv("segmentos_con_alcaldia_y_promedio.csv")

df["hora_exacta"] = pd.to_datetime(df["hora_exacta"])
df["hora_decimal"] = (df["hora_exacta"].dt.hour + df["hora_exacta"].dt.minute / 60)
df["tiempo_fisico"] = (df["distancia_km"] / df["velocidad"]) * 60
df["hora_pico"] = df["hora_decimal"].apply(lambda x: 1 if ((6 <= x <= 9) or (17 <= x <= 20)) else 0)

q1 = df["tiempo_segmento"].quantile(0.25)
q3 = df["tiempo_segmento"].quantile(0.75)
df = df[df["tiempo_segmento"] <= (q3 + 1.5 * (q3 - q1))]

df["sentido"] = df["sentido"].map({"Ida": 0, "Vuelta": 1})
df = pd.get_dummies(df, columns=["segmento_id", "dia_semana"], drop_first=True)

features = [c for c in df.columns if c not in ["tiempo_segmento", "hora_exacta", "Hora"]]
X = df[features]
y = df["tiempo_segmento"]

def crear_modelo(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

kf = KFold(n_splits=10, shuffle=True, random_state=SEED)
resultados = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = crear_modelo(X_train_scaled.shape[1])
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test),
              epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)

    pred = model.predict(X_test_scaled, verbose=0).flatten()
    resultados.append({'MAE': mean_absolute_error(y_test, pred),
                       'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
                       'R2': r2_score(y_test, pred)})
    print(f"Fold {fold} completado.")

res_df = pd.DataFrame(resultados)
print("\nRESULTADOS PROMEDIO MLP:\n", res_df.mean())

plt.figure(figsize=(10, 4))
plt.plot(res_df.index + 1, res_df['MAE'], marker='o', label='MAE por Fold')
plt.axhline(res_df['MAE'].mean(), color='r', linestyle='--', label='Promedio')
plt.title("Estabilidad del Modelo MLP (MAE)")
plt.legend()
plt.show()

scaler_final = StandardScaler()
X_scaled = scaler_final.fit_transform(X)
model_final = crear_modelo(X_scaled.shape[1])
model_final.fit(X_scaled, y, epochs=50, batch_size=32, verbose=0)
y_pred_final = model_final.predict(X_scaled).flatten()

plt.figure(figsize=(8,8))
plt.scatter(y, y_pred_final, alpha=0.3)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel("Tiempo Real")
plt.ylabel("Tiempo Predicho")
plt.title("MLP: Real vs Predicho")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import joblib
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import regularizers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

df = pd.read_csv("segmentos_con_alcaldia_y_promedio.csv")

df["hora_exacta"] = pd.to_datetime(df["hora_exacta"])
df["hora_decimal"] = (df["hora_exacta"].dt.hour + df["hora_exacta"].dt.minute / 60)
df["tiempo_fisico"] = (df["distancia_km"] / df["velocidad"]) * 60
df["hora_pico"] = df["hora_decimal"].apply(lambda x: 1 if ((6 <= x <= 9) or (17 <= x <= 20)) else 0)

q1 = df["tiempo_segmento"].quantile(0.25)
q3 = df["tiempo_segmento"].quantile(0.75)
df = df[df["tiempo_segmento"] <= (q3 + 1.5 * (q3 - q1))]

df["sentido"] = df["sentido"].map({"Ida": 0, "Vuelta": 1})
df = pd.get_dummies(df, columns=["segmento_id", "dia_semana"], drop_first=True)

features = [c for c in df.columns if c not in ["tiempo_segmento", "hora_exacta", "Hora"]]
X = df[features]
y = df["tiempo_segmento"]

def crear_modelo(input_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.Dense(1)])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

kf = KFold(n_splits=10, shuffle=True, random_state=SEED)
resultados = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = crear_modelo(X_train_scaled.shape[1])
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    history = model.fit(X_train_scaled,y_train,validation_data=(X_test_scaled, y_test), epochs=100, batch_size=32, callbacks=[early_stop], verbose=0)

    pred = model.predict(X_test_scaled, verbose=0).flatten()
    resultados.append({'MAE': mean_absolute_error(y_test, pred),
                       'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
                       'R2': r2_score(y_test, pred)})
    print(f"Fold {fold} completado.")
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.xlabel("Épocas")
plt.ylabel("MSE")
plt.title("Curva de Aprendizaje - Loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['mae'], label='Entrenamiento')
plt.plot(history.history['val_mae'], label='Validación')
plt.xlabel("Épocas")
plt.ylabel("MAE")
plt.title("Curva de Aprendizaje - MAE")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
res_df = pd.DataFrame(resultados)
print("\nRESULTADOS PROMEDIO MLP:\n", res_df.mean())
print("\nDESVIACIÓN ESTÁNDAR:\n")
print(res_df.std())

**Random Forest - Kfold Validation 10**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("segmentos_con_alcaldia_y_promedio.csv")
df["hora_exacta"] = pd.to_datetime(df["hora_exacta"])
df["hora_decimal"] = (df["hora_exacta"].dt.hour + df["hora_exacta"].dt.minute / 60)
df["sentido"] = df["sentido"].map({"Ida": 0, "Vuelta": 1})

df = pd.get_dummies(df, columns=["segmento_id", "dia_semana"], drop_first=True)

elimina = ["hora_exacta", "tiempo_segmento", "tiempo_acumulado_viaje", "vel_promedio_movimiento", "Hora"]
variables = [c for c in df.columns if c not in elimina]

X = df[variables]
y = df["tiempo_segmento"]

kf = KFold(n_splits=10, shuffle=True, random_state=42)
resultados = []
rf_params = {'n_estimators': 200, 'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 2, 'random_state': 42, 'n_jobs': -1}

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    modelo_rf = RandomForestRegressor(**rf_params)
    modelo_rf.fit(X_train, y_train)
    pred = modelo_rf.predict(X_test)

    resultados.append({
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2': r2_score(y_test, pred)})


resultados_df = pd.DataFrame(resultados)
print("\nRESULTADOS PROMEDIO Bosque AleatorioT:\n", resultados_df.mean())

modelo_final = RandomForestRegressor(**rf_params).fit(X, y)
y_pred_final = modelo_final.predict(X)

plt.figure(figsize=(8,8))
plt.scatter(y, y_pred_final, alpha=0.3, color='green')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.title("Random Forest: Real vs Predicho")
plt.xlabel("Tiempo Real")
plt.ylabel("Tiempo Predicho")
plt.show()


importancias = pd.DataFrame({'Variable': features, 'Importancia': modelo_final.feature_importances_})
print(importancias.sort_values(by='Importancia', ascending=False).head(10))

top15 = importancias.head(15)
plt.figure(figsize=(10,6))
plt.barh(top15["Variable"], top15["Importancia"])
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.title("Importancia de Variables - Random Forest")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Guardar
joblib.dump(modelo_final, "modelo_rf_final.pkl")

**Gráficas de medidas de desempeño de los modelos RLM, MLP, RF**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

modelos = ['Regresión Lineal',
           'Perceptrón Multicapa',
           'Bosque Aleatorio']

mae = [0.28, 0.13, 0.10]
rmse = [0.43, 0.17, 0.20]
r2 = [0.82, 0.96, 0.95]

colores = ['steelblue', 'darkorange', 'seagreen']

plt.figure(figsize=(8,5))
plt.bar(modelos, mae, color=colores)
plt.title('Comparación de MAE')
plt.ylabel('MAE')
plt.xlabel('Modelo')
plt.grid(axis='y', alpha=0.3)
plt.show()

plt.figure(figsize=(8,5))
plt.bar(modelos, rmse, color=colores)
plt.title('Comparación de RMSE')
plt.ylabel('RMSE')
plt.xlabel('Modelo')
plt.grid(axis='y', alpha=0.3)
plt.show()

plt.figure(figsize=(8,5))
plt.bar(modelos, r2, color=colores)
plt.title('Comparación de R²')
plt.ylabel('R²')
plt.xlabel('Modelo')
plt.grid(axis='y', alpha=0.3)
plt.show()

**Tiempo de entrenamiento de los modelos entreandos RLM, MLP, RF**

In [ ]:
import time
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

start_time = time.time()
reg = LinearRegression().fit(X_scaled, y)
time_lr = time.time() - start_time


start_time = time.time()
rf = RandomForestRegressor(n_estimators=200, n_jobs=-1).fit(X, y)
time_rf = time.time() - start_time

start_time = time.time()
mlp = crear_modelo(X_scaled.shape[1])
mlp.fit(X_scaled, y, epochs=50, batch_size=16, verbose=0)
time_mlp = time.time() - start_time

print(f"Tiempo Entrenamiento Regresión Lineal : {time_lr:.4f} segundos")
print(f"Tiempo Entrenamiento Random Forest    : {time_rf:.4f} segundos")
print(f"Tiempo Entrenamiento Red Neuronal     : {time_mlp:.4f} segundos")

**ETA entre estaciones**

In [ ]:
import pandas as pd
import numpy as np
import joblib
from datetime import datetime, timedelta

modelo = joblib.load("modelo_rf_final.pkl")
features_modelo = list(modelo.feature_names_in_)
print("Modelo cargado correctamente.\n")

historico = pd.read_csv("segmentos_con_alcaldia_y_promedio.csv")
segmentos = pd.read_csv("segmentosMo.csv")
estaciones = pd.read_csv("Estaciones.csv")
print("Archivos cargados correctamente.\n")

historico["hora_exacta"] = pd.to_datetime(historico["hora_exacta"])
historico["hora_decimal"] = (
    historico["hora_exacta"].dt.hour +
    historico["hora_exacta"].dt.minute/60)
historico["sentido"] = historico["sentido"].map({"Ida":0,"Vuelta":1})

ahora = datetime.now()
hora_decimal = ahora.hour + ahora.minute/60
mes_actual = ahora.month
anio_actual = ahora.year
dias = {
    0:"Lunes",
    1:"Martes",
    2:"Miércoles",
    3:"Jueves",
    4:"Viernes",
    5:"Sábado",
    6:"Domingo"}
dia_actual = dias[ahora.weekday()]
fin_semana = 1 if ahora.weekday() >= 5 else 0
print("Fecha del sistema")
print(ahora.strftime("%d/%m/%Y %H:%M"))

print(" ESTIMACIÓN DE TIEMPO DE ARRIBO")
origen = input("Estación origen : ").strip()
destino = input("Estación destino: ").strip()
sentido_usuario = input("Sentido (Ida/Vuelta): ").strip().capitalize()

lista_estaciones = estaciones["Nombre"].tolist()
if origen not in lista_estaciones:
    raise ValueError(f"La estación '{origen}' no existe.")
if destino not in lista_estaciones:
    raise ValueError(f"La estación '{destino}' no existe.")
if sentido_usuario not in ["Ida","Vuelta"]:
    raise ValueError("El sentido debe ser Ida o Vuelta.")

if sentido_usuario == "Ida":
    inicio = segmentos[segmentos["estacion_inicio"] == origen].index[0]
    fin = segmentos[segmentos["estacion_fin"] == destino].index[0]
    if inicio > fin:
        raise ValueError("El origen y destino no corresponden al sentido Ida.")
    ruta = segmentos.iloc[inicio:fin+1].copy()
else:
    segmentos_rev = segmentos.iloc[::-1].reset_index(drop=True)
    inicio = segmentos_rev[segmentos_rev["estacion_fin"] == origen].index[0]
    fin = segmentos_rev[segmentos_rev["estacion_inicio"] == destino].index[0]
    if inicio > fin:
        raise ValueError("El origen y destino no corresponden al sentido Vuelta.")
    ruta = segmentos_rev.iloc[inicio:fin+1].copy()

distancia_total = ruta["distancia_km"].sum()
print("Ruta encontrada correctamente")
print(f"Origen             : {origen}")
print(f"Destino            : {destino}")
print(f"Sentido            : {sentido_usuario}")
print(f"Número de segmentos: {len(ruta)}")
print(f"Distancia total    : {distancia_total:.2f} km")
tiempo_total = 0
detalle = []
for _, segmento in ruta.iterrows():
    id_segmento = segmento["segmento_id"]
    candidatos = historico[
        (historico["segmento_id"] == id_segmento) &
        (historico["sentido"] == (0 if sentido_usuario=="Ida" else 1))].copy()
    if "dia_semana" in candidatos.columns:
        mismo_dia = candidatos[candidatos["dia_semana"] == dia_actual]
        if len(mismo_dia) > 0:
            candidatos = mismo_dia

    if len(candidatos) == 0:
        candidatos = historico[
            historico["segmento_id"] == id_segmento].copy()

    candidatos["dif_hora"] = abs(candidatos["hora_decimal"] - hora_decimal)
    registro = candidatos.sort_values("dif_hora").iloc[0]

    entrada = {}
    for variable in features_modelo:
        entrada[variable] = 0
    for columna in ["viaje_id",
        "lat1",
        "lon1",
        "lat2",
        "lon2",
        "distancia_km",
        "velocidad",
        "anio",
        "mes_num",
        "hora_decimal",
        "es_fin_de_semana"]:
        if columna in entrada:
            if columna == "anio":
                entrada[columna] = anio_actual
            elif columna == "mes_num":
                entrada[columna] = mes_actual
            elif columna == "hora_decimal":
                entrada[columna] = hora_decimal
            elif columna == "es_fin_de_semana":
                entrada[columna] = fin_semana
            else:
                entrada[columna] = registro[columna]

    if "sentido" in entrada:
        entrada["sentido"] = 0 if sentido_usuario=="Ida" else 1

    columna_segmento = f"segmento_id_{int(id_segmento)}"
    if columna_segmento in entrada:
        entrada[columna_segmento] = 1

    columna_dia = f"dia_semana_{dia_actual}"
    if columna_dia in entrada:
        entrada[columna_dia] = 1

    entrada_df = pd.DataFrame([entrada])
    entrada_df = entrada_df[features_modelo]

    tiempo = modelo.predict(entrada_df)[0]
    tiempo_total += tiempo
    detalle.append({"Inicio": segmento["estacion_inicio"],
        "Fin": segmento["estacion_fin"],
        "Tiempo": tiempo})

hora_llegada = ahora + timedelta(minutes=float(tiempo_total))

print()
print("RESULTADO DE LA ESTIMACIÓN")
print(f"Origen              : {origen}")
print(f"Destino             : {destino}")
print(f"Sentido             : {sentido_usuario}")
print(f"Fecha               : {ahora.strftime('%d/%m/%Y')}")
print(f"Hora de salida      : {ahora.strftime('%H:%M')}")
print(f"Hora estimada llegada: {hora_llegada.strftime('%H:%M')}")
print(f"Distancia total     : {distancia_total:.2f} km")
print(f"Segmentos recorridos: {len(ruta)}")
print(f"Tiempo estimado     : {tiempo_total:.2f} minutos")
print()
